In [35]:
import warnings

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
)

## Enable if upsampling is used 
# from sklearn.utils import resample 
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
import pandas as pd
import numpy as np
import os
import torch
import json


## GPU/RAM Check

In [5]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Fri Mar  6 11:14:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   33C    P0             48W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [6]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 189.9 gigabytes of available RAM

You are using a high-RAM runtime!


## Preparing Training Data

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
cleaned_sentment_dataset = pd.read_pickle('/content/drive/MyDrive/ColabThesisData/finnsentiment_cleaned_unambiguous.pkl')

# Replace with your dataset loading
ds = cleaned_sentment_dataset

In [9]:
ds.head(10)

,label,text
2,1,40.
3,2,Kyseessä voi olla loppuelämäsi nainen.
4,2,Sinne vaan ocean clubiin iskemään!
5,2,Itsekin pidän Keskustan kampanjointia ihan hyv...
6,1,"Kamppi, Kontula, Kluuvi"
8,0,en haluaisi että kissani vuotaa.. =)
9,0,Nyt olisi lääkitys paikallaan.
10,1,Ihmiset joutuvat joskus monenlaisien päätelmie...
11,0,"Eniten pelkään sitä, että jos mies vain koko a..."
12,2,Muutenkin suosittelen kaikille asiasta kiinnos...


In [10]:
ds["label"].value_counts()

,count
label,
1,12374
2,1358
0,1230


In [11]:
df_0 = ds[ds['label'] == 0]
df_1 = ds[ds['label'] == 1]
df_2 = ds[ds['label'] == 2]

# Undersample class 1
df_1_balanced = df_1.sample(n=min(len(df_0), len(df_2)), random_state=42)

# Combine for balanced dataset
balanced_ds = pd.concat([df_0, df_1_balanced, df_2])

In [12]:
# Upsample classes 0 and 2
# df_0_upsampled = resample(df_0, replace=True, n_samples=len(df_1), random_state=42)
# df_2_upsampled = resample(df_2, replace=True, n_samples=len(df_1), random_state=42)

# balanced_ds = pd.concat([df_1, df_0_upsampled, df_2_upsampled])

In [13]:
balanced_ds["label"].value_counts()

,count
label,
2,1358
0,1230
1,1230


In [14]:
balanced_ds

,label,text
8,0,en haluaisi että kissani vuotaa.. =)
9,0,Nyt olisi lääkitys paikallaan.
11,0,"Eniten pelkään sitä, että jos mies vain koko a..."
17,0,Tuntuu kuin olisin pelkkä huora.
22,0,Missä kohtaa olen sinua nimitellyt?
...,...,...
23624,2,"Siitä huolimatta naimme salassa, jotta hyvä tu..."
23633,2,Hyvää ystävänpäivää
23637,2,Kiitos kun nostatit hymyn huulilleni jälleen :)
23758,2,Paluuhali Sinulle;)


## Define a Common Config

In [ ]:
# Configuration
class Config:
    # Model settings
    model_name = "TurkuNLP/finnish-modernbert-large"  # Use base for resource efficiency
        
    # Training settings
    max_seq_length = 4096 # Blackwell can handle long contexts easily
    batch_size = 16  # Large batch size for Blackwell (100GB VRAM)
    learning_rate = 2e-4  # Higher than standard fine-tuning due to LoRA
    num_epochs = 5 # Train longer with powerful hardware
    warmup_steps = 100
    weight_decay = 0.01
    logging_steps = 10
    save_steps = 500
    eval_steps = 500
    
    # Data settings
    num_labels = 3  # negative, neutral, positive
    label_names = ["negative", "neutral", "positive"]
    
    # Output settings
    output_dir = "/content/drive/MyDrive/ColabThesisData/"
    logging_dir = "./logs"
    
config = Config()

## Model and Tokenizer Setup

In [16]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name)

# Load model with quantization
model = AutoModelForSequenceClassification.from_pretrained(
    config.model_name,
    num_labels=config.num_labels,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

# Print model info
print(f"Model loaded: {config.model_name}")
print(f"Model parameters: {model.num_parameters():,}")
print(f"Model device: {next(model.parameters()).device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: TurkuNLP/finnish-modernbert-large
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: TurkuNLP/finnish-modernbert-large
Model parameters: 401,208,323
Model device: cuda:0


## Tokenize and Split Dataset for Training and Evaluation

In [17]:
# Split your DataFrame
train_df, temp_df = train_test_split(balanced_ds, test_size=0.3, random_state=42, stratify=balanced_ds['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Preprocess using map (not pandas apply)
def preprocess(batch):
    enc = tokenizer(batch["text"], truncation=True, max_length=config.max_seq_length)
    enc["label"] = batch["label"]
    return enc

train_dataset = train_dataset.map(preprocess, batched=True)
val_dataset = val_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)

# Create DatasetDict for Trainer
ds_preprocessed = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

Map:   0%|          | 0/2672 [00:00<?, ? examples/s]

Map:   0%|          | 0/573 [00:00<?, ? examples/s]

Map:   0%|          | 0/573 [00:00<?, ? examples/s]

## Training and Evaluation Setup

In [23]:
def compute_metrics(eval_pred):
    """Compute accuracy and other metrics"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    recall = recall_score(y_true=labels, y_pred=predictions, average="weighted")
    precision = precision_score(y_true=labels, y_pred=predictions, average="weighted")
    f1 = f1_score(y_true=labels, y_pred=predictions, average="weighted")

    return {
        'accuracy': accuracy,
        'recall': recall,
        'precision': precision,
        'f1': f1
    }

# Force PyTorch to use the modern API for TF32
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Training arguments
training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_epochs,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    warmup_steps=config.warmup_steps,
    logging_dir=config.logging_dir,
    logging_steps=config.logging_steps,
    eval_strategy="steps",
    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    dataloader_pin_memory=True,
    gradient_checkpointing=False,  
    fp16=False,  
    bf16=torch.cuda.is_available(), # Use bfloat16 if available
    tf32=torch.cuda.is_available(),  
    report_to="none",  # Disable wandb for now
    remove_unused_columns=True,
    torch_compile=True,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [24]:
def predict_sentiment(text, model, tokenizer):
    """Predict sentiment for a single text"""
    model.eval()
    
    # Tokenize
    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=config.max_seq_length,
        return_tensors='pt'
    )
    
    # Move to device
    input_ids = encoding['input_ids'].to(model.device)
    attention_mask = encoding['attention_mask'].to(model.device)
    
    # Predict
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=-1)
        predicted_class = torch.argmax(logits, dim=-1).item()
    
    return {
        'predicted_label': config.label_names[predicted_class],
        'predicted_class': predicted_class,
        'probabilities': {
            label: prob.item() 
            for label, prob in zip(config.label_names, probabilities[0])
        },
        'confidence': probabilities[0][predicted_class].item()
    }


## Main Training Pipeline

In [25]:
def main_training_pipeline():
    # """Main training pipeline"""
    
    print("=" * 50)
    print("STARTING MODERNBERT TUNING FOR DOWNSTREAM SENTIMENT ANALYSIS")
    print("=" * 50)

    # Create data collator
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
        
    # 5. Create trainer
    print("\n5. Creating trainer...")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        data_collator=data_collator,  
    )
    
    
    # 6. Start training
    print("\n6. Starting training...")
    print(f"Training for {config.num_epochs} epochs")
    print(f"Effective batch size: {config.batch_size * config.gradient_accumulation_steps}")

    # Train!
    trainer.train()
    
    # 7. Save final model
    print("\n7. Saving final model...")
    final_model_path = os.path.join(config.output_dir, "final_model")
    trainer.save_model(final_model_path)
    tokenizer.save_pretrained(final_model_path)
    
    # 8. Evaluate on test set
    print("\n8. Evaluating on test set...")
    predictions, label_ids, metrics = trainer.predict(test_dataset)
    print(f"Test Results: {metrics}")
    
    print("\n" + "=" * 50)
    print("TRAINING COMPLETED SUCCESSFULLY!")
    print(f"Model saved to: {final_model_path}")
    print("=" * 50)
    
    return model, tokenizer, metrics

## Define Utility Functions

In [26]:
def save_config():
    """Save configuration to JSON"""
    config_dict = {k: v for k, v in config.__dict__.items() if not k.startswith('_')}
    
    with open(os.path.join(config.output_dir, 'training_config.json'), 'w') as f:
        json.dump(config_dict, f, indent=2)

def load_trained_model(model_path):
#     """Load a trained model"""

    
    model = AutoModelForSequenceClassification.from_pretrained(
        config.model_name,
        num_labels=config.num_labels,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    )
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    return model, tokenizer

## Execute the Pipeline

In [ ]:
# Create output directory
os.makedirs(config.output_dir, exist_ok=True)

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore", message=".*Online softmax is disabled.*")

# Save configuration
save_config()

# Run training
model, tokenizer, test_results = main_training_pipeline()

# Test prediction on sample texts
print("\n" + "=" * 50)
print("TESTING PREDICTIONS")
print("=" * 50)

sample_texts = [
    "The company's quarterly earnings exceeded all expectations with strong revenue growth.",
    "Stock prices fell dramatically due to market uncertainty and economic concerns.",
    "The financial report shows stable performance with no significant changes."
]

for text in sample_texts:
    result = predict_sentiment(text, model, tokenizer)
    print(f"\nText: {text}")
    print(f"Predicted: {result['predicted_label']} (confidence: {result['confidence']:.3f})")
    print(f"Probabilities: {result['probabilities']}")

STARTING MODERNBERT TUNING FOR DOWNSTREAM SENTIMENT ANALYSIS

5. Creating trainer...

6. Starting training...
Training for 5 epochs
Effective batch size: 32


/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7627: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7627: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(


Step,Training Loss,Validation Loss


/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7627: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


7. Saving final model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


8. Evaluating on test set...


Test Results: {'test_loss': 0.43668460845947266, 'test_accuracy': 0.9197207678883071, 'test_recall': 0.9197207678883071, 'test_precision': 0.9197215376034047, 'test_f1': 0.9196134596800947, 'test_runtime': 2.0623, 'test_samples_per_second': 277.846, 'test_steps_per_second': 17.456}

TRAINING COMPLETED SUCCESSFULLY!
Model saved to: /content/drive/MyDrive/ColabThesisData/final_model

TESTING PREDICTIONS

Text: The company's quarterly earnings exceeded all expectations with strong revenue growth.
Predicted: neutral (confidence: 0.615)
Probabilities: {'negative': 0.0001462948421249166, 'neutral': 0.6149978637695312, 'positive': 0.38485586643218994}

Text: Stock prices fell dramatically due to market uncertainty and economic concerns.
Predicted: neutral (confidence: 0.984)
Probabilities: {'negative': 0.015905803069472313, 'neutral': 0.9840571284294128, 'positive': 3.703780748764984e-05}

Text: The financial report shows stable performance with no significant changes.
Predicted: neutral (con

In [28]:
predict_sentiment("nokia julkaisi tänä vuonna positiivisen tulosvaroituksen", model, tokenizer)

{'predicted_label': 'neutral',
 'predicted_class': 1,
 'probabilities': {'negative': 4.824161692340567e-07,
  'neutral': 0.9868782162666321,
  'positive': 0.013121264055371284},
 'confidence': 0.9868782162666321}

In [29]:
predict_sentiment("nokian tulosraportti ylitti analyytikoiden odotukset", model, tokenizer)

{'predicted_label': 'neutral',
 'predicted_class': 1,
 'probabilities': {'negative': 6.529138772748411e-06,
  'neutral': 0.9982660412788391,
  'positive': 0.0017274473793804646},
 'confidence': 0.9982660412788391}

In [30]:
predict_sentiment("tää firma menee kohta konkurssiin", model, tokenizer)

{'predicted_label': 'negative',
 'predicted_class': 0,
 'probabilities': {'negative': 0.9998373985290527,
  'neutral': 0.00016156042693182826,
  'positive': 1.1014175242962665e-06},
 'confidence': 0.9998373985290527}

In [31]:
predict_sentiment("tämä on todella hyvä uutinen", model, tokenizer)

{'predicted_label': 'positive',
 'predicted_class': 2,
 'probabilities': {'negative': 8.327813993957989e-10,
  'neutral': 3.850741947530878e-09,
  'positive': 1.0},
 'confidence': 1.0}

In [32]:
predict_sentiment("Olisiko tässä mun eka IPO johon osallistuisisin", model, tokenizer)

{'predicted_label': 'neutral',
 'predicted_class': 1,
 'probabilities': {'negative': 2.1477712053297182e-08,
  'neutral': 0.9999998807907104,
  'positive': 1.3574332058396976e-07},
 'confidence': 0.9999998807907104}

In [33]:
predict_sentiment("Eihän Aallon groupilla ole omaa softaa, jota myydään asiakkaille?", model, tokenizer)

{'predicted_label': 'neutral',
 'predicted_class': 1,
 'probabilities': {'negative': 2.486225669784403e-09,
  'neutral': 1.0,
  'positive': 1.4166087192180044e-09},
 'confidence': 1.0}

In [34]:
predict_sentiment("Näkymien ennustamisessa mielestäni sinällään vastuullinen ennuste, jos korona iskee ennusteita isommin niin skeptisen varovainen mielummin kuin “rotsi auki”. Ilman tätä pöpöä tässä olisi hieman toisenlainen arvostustaso", model, tokenizer)

{'predicted_label': 'neutral',
 'predicted_class': 1,
 'probabilities': {'negative': 0.00046280008973553777,
  'neutral': 0.9995191097259521,
  'positive': 1.8050159269478172e-05},
 'confidence': 0.9995191097259521}

In [36]:
predict_sentiment("Nokia on hyvä sijoitus pitkällä aikavälillä", model, tokenizer)

{'predicted_label': 'positive',
 'predicted_class': 2,
 'probabilities': {'negative': 2.4858781699776955e-09,
  'neutral': 0.0001398220774717629,
  'positive': 0.9998601675033569},
 'confidence': 0.9998601675033569}

In [37]:
predict_sentiment("Huh huh, nyt kyllä jysähti!", model, tokenizer)

{'predicted_label': 'negative',
 'predicted_class': 0,
 'probabilities': {'negative': 0.9585275053977966,
  'neutral': 1.0664319233910646e-05,
  'positive': 0.04146183282136917},
 'confidence': 0.9585275053977966}

In [38]:
predict_sentiment("Onko omaa kokemusta Hongan tämän päivän tuotteista?", model, tokenizer)

{'predicted_label': 'neutral',
 'predicted_class': 1,
 'probabilities': {'negative': 8.592165867682411e-10,
  'neutral': 0.9999998807907104,
  'positive': 1.2359568302144908e-07},
 'confidence': 0.9999998807907104}

In [39]:
predict_sentiment("Taidan lisätä huomenna avauksesta. Hyvältä vaikuttaa tämä case!", model, tokenizer)

{'predicted_label': 'positive',
 'predicted_class': 2,
 'probabilities': {'negative': 5.211409082050977e-10,
  'neutral': 5.49606625099841e-07,
  'positive': 0.9999994039535522},
 'confidence': 0.9999994039535522}